In [ ]:
import math
import json
import time
import torch
import pydantic
import src

assert src.__version__ == "0.1.0"
print("Environment Verified:", src.__name__, src.__version__)

In [ ]:
from src.environment.mock_data import get_mock_database
from src.environment.sandbox import SandboxedToolEngine
from src.environment.tools import TOOL_CONTRACTS

db = get_mock_database(42)
assert len(db["vendors"]) == 12
assert len(db["inventory_items"]) == 20
assert len(db["purchase_orders"]) == 30
assert len(db["budget_caps"]) == 4

engine = SandboxedToolEngine(seed=42)
obs_lookup = engine.execute("lookup_records", {"table": "vendors", "filters": {"category": "Hardware"}})
assert not obs_lookup.is_error
assert len(obs_lookup.result) > 0

obs_agg = engine.execute("compute_aggregate", {"values": [100.0, 250.0, 50.0], "operation": "sum"})
assert not obs_agg.is_error
assert obs_agg.result == 400.0

obs_verify = engine.execute("verify_constraint", {"actual": 400.0, "expected": 500.0, "comparison": "le"})
assert not obs_verify.is_error
assert obs_verify.result is True

print("Tool Engine Verified:", list(TOOL_CONTRACTS.keys()))

In [ ]:
from src.environment.task_generator import generate_dataset_splits, verify_split_isolation

splits = generate_dataset_splits(seed=42, train_count=20, val_count=10, test_count=10)
assert len(splits["train"]) == 20
assert len(splits["val"]) == 10
assert len(splits["test"]) == 10

is_isolated, issues = verify_split_isolation(splits)
assert is_isolated
assert len(issues) == 0

print("Split Isolation Verified: Zero Leakage Detected")

In [ ]:
from src.data.redaction import redact_text, redact_data

secret_sample = "User admin@lexsi.corp with token Bearer sec12345678 and api_key=sk-test98765432"
cleaned = redact_text(secret_sample)
assert "admin@lexsi.corp" not in cleaned
assert "Bearer sec12345678" not in cleaned
assert "sk-test98765432" not in cleaned

print("Redaction Filter Verified:", cleaned)

In [ ]:
from src.runtime.orchestrator import EpisodeOrchestrator
from src.runtime.parser import parse_agent_output

task = splits["train"][0]
orchestrator = EpisodeOrchestrator(seed=42)
category = task.target_entities["category"]

call_step = 0
def oracle_policy(dialog):
    global call_step
    call_step += 1
    if call_step == 1:
        return f'<|START_ACTION|>{{"name": "lookup_records", "arguments": {{"table": "inventory_items", "filters": {{"category": "{category}"}}}}}}<|END_ACTION|>'
    if call_step == 2:
        return f'<|START_ACTION|>{{"name": "compute_aggregate", "arguments": {{"values": [{task.oracle_answer}], "operation": "sum"}}}}<|END_ACTION|>'
    return f"Audit Complete. Verified total is ${task.oracle_answer:.2f}."

trajectory = orchestrator.run_episode(task, oracle_policy)
assert trajectory.is_success
assert trajectory.total_steps == 3
assert len(trajectory.steps) == 3

print("Episode Orchestrator Verified: Task Success =", trajectory.is_success)

In [ ]:
from src.evaluation.metrics import evaluate_benchmark_run
from src.evaluation.taxonomy import classify_failure, FailureCategory

eval_res = evaluate_benchmark_run([trajectory], [task], run_id="notebook_eval_01")
assert eval_res.metrics["M1_task_success_rate"].value == 1.0
assert eval_res.metrics["M3_schema_validity"].value == 1.0
assert len(eval_res.sha256_fingerprint) == 64

print("AuditKIT Evaluation Verified:")
for metric_id, metric in eval_res.metrics.items():
    print(f"  {metric_id}: {metric.value}")